# AI Agent Loop Demo


This notebook demonstrates the core mechanism behind an AI agent: a model that can decide to call a tool, receive the result, and continue reasoning toward a goal — all in a loop.

---

This notebook contains starter code. Please fill in details as needed.
---

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to [aistudio.google.com](https://aistudio.google.com/prompts/new_chat)
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0
* [Developer guide](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)



In [4]:

!pip install google-genai --quiet
print("Done.")

Done.


In [5]:
import os

try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    print("API key loaded from Colab Secrets.")
except Exception:
    import getpass
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Paste your Gemini API key: ")
    print("API key set.")


API key loaded from Colab Secrets.


In [12]:
import os

try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    print("API key loaded from Colab Secrets.")
except Exception:
    import getpass
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Paste your Gemini API key: ")
    print("API key set.")

# %% Cell 3: recipe database
DOCUMENTS = {
    "chicken_biryani": (
    "500g chicken (bone-in or boneless). "
    "2 cups basmati rice. "
    "2 large onions, thinly sliced. "
    "2 tomatoes, chopped. "
    "1/2 cup plain yogurt. "
    "2 tbsp ginger-garlic paste. "
    "2 green chilies, slit. "
    "1/4 cup chopped mint leaves. "
    "1/4 cup chopped coriander leaves. "
    "1 tsp red chili powder. "
    "1/2 tsp turmeric powder. "
    "1 tsp coriander powder. "
    "1 tsp garam masala. "
    "1/2 tsp cumin seeds. "
    "2 bay leaves. "
    "4 cloves. "
    "2 green cardamoms. "
    "1 inch cinnamon stick. "
    "3 tbsp oil or ghee. "
    "Salt to taste. "
    "Soak the basmati rice in water for 30 minutes, then drain. "
    "Wash the chicken and marinate it with yogurt, ginger-garlic paste, chili powder, turmeric, coriander powder, garam masala, and salt for at least 30 minutes. "
    "Heat oil or ghee in a heavy-bottomed pot. Add bay leaves, cloves, cardamoms, cinnamon, and cumin seeds. "
    "Add sliced onions and fry until golden brown. Add green chilies and chopped tomatoes. Cook until the tomatoes soften. "
    "Add the marinated chicken and cook for 8-10 minutes until lightly browned. "
    "Add mint leaves and coriander leaves and mix well. "
    "In another pot, boil water with salt and cook the soaked rice until it is about 70 percent cooked. Drain the rice. "
    "Spread the partially cooked rice evenly over the chicken masala. "
    "Cover tightly and cook on low heat for 20-25 minutes until the chicken is fully cooked and the rice is tender. "
    "Let it rest for 10 minutes before opening the lid. "
    "Gently mix and serve hot with raita or boiled eggs."
),

"chicken_ghee_roast": (
    "500g chicken, preferably bone-in pieces. "
    "2 tbsp ghee. "
    "1/2 cup plain yogurt. "
    "1 tbsp ginger-garlic paste. "
    "1 tsp turmeric powder. "
    "1 tsp tamarind paste. "
    "1 tsp jaggery. "
    "Salt to taste. "
    "For the masala: 8 dried Kashmiri red chilies. "
    "1 tsp coriander seeds. "
    "1/2 tsp cumin seeds. "
    "1/2 tsp fennel seeds. "
    "1/4 tsp black peppercorns. "
    "4 cloves. "
    "1 small cinnamon stick. "
    "Marinate the chicken with yogurt, ginger-garlic paste, turmeric powder, and salt for at least 30 minutes. "
    "Dry roast the dried red chilies, coriander seeds, cumin seeds, fennel seeds, peppercorns, cloves, and cinnamon on low heat until fragrant. "
    "Allow the roasted spices to cool and grind them into a smooth paste using a little water. "
    "Heat 1 tbsp ghee in a pan and add the ground masala paste. Cook on low heat until the raw smell disappears and the masala thickens. "
    "Add tamarind paste and jaggery and mix well. "
    "Add the marinated chicken and cook for 10 minutes, stirring occasionally. "
    "Cover and cook on medium-low heat until the chicken is fully cooked and tender. "
    "Add the remaining ghee and roast the chicken on medium heat until the masala becomes thick, glossy, and coats the chicken pieces. "
    "Cook until the ghee separates and the masala turns dark reddish-brown. "
    "Garnish with curry leaves and serve hot with neer dosa, rice, or chapati."
),

"masala_dosa": (
    "2 cups dosa rice. "
    "1/2 cup urad dal. "
    "1/2 tsp fenugreek seeds. "
    "1/2 cup poha. "
    "Salt to taste. "
    "Water as needed. "
    "Oil or ghee for cooking. "
    "For the potato masala: 3 medium potatoes, boiled and mashed. "
    "2 onions, thinly sliced. "
    "2 green chilies, chopped. "
    "1/2 tsp mustard seeds. "
    "1/2 tsp cumin seeds. "
    "1/2 tsp turmeric powder. "
    "8 curry leaves. "
    "2 tbsp chopped coriander leaves. "
    "1 tbsp oil. "
    "Salt to taste. "
    "Wash and soak the dosa rice and urad dal separately for 4-6 hours. Add fenugreek seeds to the urad dal while soaking. "
    "Soak poha for 15 minutes before grinding. "
    "Grind the urad dal, rice, fenugreek seeds, and poha separately into a smooth batter using water as needed. "
    "Mix everything together, add salt, and allow the batter to ferment for 8-12 hours or overnight until it rises. "
    "For the potato masala, heat oil in a pan and add mustard seeds and cumin seeds. Allow them to splutter. "
    "Add curry leaves, green chilies, and sliced onions. Saute until the onions turn soft. "
    "Add turmeric powder, mashed potatoes, salt, and a little water. Mix well and cook for 3-4 minutes. "
    "Add chopped coriander leaves and mix. Keep the potato masala aside. "
    "Heat a dosa tawa and lightly grease it with oil. Pour a ladle of fermented batter and spread it into a thin circle. "
    "Drizzle oil or ghee around the edges and cook until the bottom turns golden and crispy. "
    "Place a portion of potato masala in the center and fold the dosa. "
    "Serve hot with coconut chutney and sambar."
),
}

API key loaded from Colab Secrets.


In [7]:
import re

STOPWORDS = {
    "a", "an", "the", "and", "or", "of", "to", "in", "on", "for", "with", "is",
    "are", "how", "what", "which", "do", "does", "can", "i", "we", "our", "it",
    "long", "much", "many", "need", "needed", "make", "ingredients", "recipe",
}


def search_documents(query: str) -> str:
    """Search the cookbook for recipes relevant to the query."""
    words = {w for w in re.findall(r"[a-z]+", query.lower()) if w not in STOPWORDS}
    results = []

    for name, content in DOCUMENTS.items():
        title_words = set(name.replace("_", " ").split())
        text_words = set(re.findall(r"[a-z]+", content.lower()))
        score = sum(3 for w in words if w in title_words)   # title match counts more
        score += sum(1 for w in words if w in text_words)   # whole-word content match
        if score > 0:
            results.append((score, name, content))

    results.sort(key=lambda r: r[0], reverse=True)

    if not results:
        return "No relevant recipes found for that query."

    return "\n\n".join(f"[Source: {n}]\n{c}" for _, n, c in results[:2])


print(search_documents("golibaje"))


[Source: golibaje]
1 cup maida (all-purpose flour). 1/2 cup curd. 2 tablespoons rice flour. 1 tablespoon sugar. 1/2 teaspoon cumin seeds. 1/2 teaspoon baking soda. 1 green chilli finely chopped. 1 tablespoon chopped coriander. 1/2 teaspoon grated ginger. Salt to taste. Water as required. Oil for deep frying. Mix maida rice flour curd sugar cumin seeds baking soda and salt in a bowl. Add chopped green chilli ginger and coriander. Add water little by little and mix to make a thick sticky batter. Cover the batter and let it rest for 2 to 3 hours. Heat oil in a deep pan over medium heat. Wet your hands, take small portions of the batter and gently drop them into the hot oil. Deep fry until the golibaje become golden brown and crispy. Drain on a paper towel and serve hot with coconut chutney.


In [8]:
import time
from google import genai
from google.genai import types

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

# NOTE: gemini-2.5-flash returns 404 for new users; gemini-3.8-flash is the current model.
# To list models available to your key:
#   for m in client.models.list(): print(m.name)
MODEL_CANDIDATES = ["gemini-3.8-flash", "gemini-3.5-flash-lite"]  # tried in order

search_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="search_documents",
            description="Search the Coastal Karnataka cookbook for relevant recipes.",
            parameters={
                "type": "object",
                "properties": {"query": {"type": "string"}},
                "required": ["query"],
            },
        )
    ]
)

config = types.GenerateContentConfig(
    system_instruction=(
        "You are a chef training at a coastal Karnataka restaurant in Mangaluru. "
        "Use search_documents for questions about ingredients, recipes, cooking "
        "times, and preparation methods. Answer only using the retrieved documents. "
        "If the documents do not contain the answer, say so clearly."
    ),
    tools=[search_tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)

TOOLS = {"search_documents": search_documents}


In [9]:
def send_with_retry(contents, max_retries=3):
    last_error = None
    for model in MODEL_CANDIDATES:
        for attempt in range(max_retries):
            try:
                return client.models.generate_content(
                    model=model, contents=contents, config=config
                )
            except Exception as e:
                err = str(e)
                last_error = e
                if "404" in err:                       # model not available -> next model
                    break
                if not any(c in err for c in ("429", "500", "502", "503", "504")):
                    raise                              # a real bug, don't hide it
                wait = 15 * (attempt + 1)
                print(f"{model}: temporary error ({err[:120]}). Retrying in {wait}s...")
                time.sleep(wait)
        print(f"Switching away from {model}...")
    raise last_error


def run_agent(question: str, max_steps: int = 8) -> str:
    print(f"Question: {question}\n" + "-" * 50)
    contents = [types.Content(role="user", parts=[types.Part(text=question)])]

    for step in range(1, max_steps + 1):
        response = send_with_retry(contents)

        if not response.candidates or not response.candidates[0].content:
            print("No response returned by Gemini.")
            return ""

        model_content = response.candidates[0].content
        contents.append(model_content)   # keep the model turn unmodified

        calls = [p.function_call for p in (model_content.parts or []) if p.function_call]

        if not calls:
            answer = response.text or "No text answer returned."
            print(f"\nFinal answer:\n{answer}")
            return answer

        result_parts = []
        for call in calls:
            func = TOOLS.get(call.name)
            if func is None:
                result = "Unknown tool."
            else:
                args = dict(call.args)
                print(f"Step {step}: {call.name}({args})")
                result = func(**args)
                print(f"  Result preview: {result[:100]}...")

            result_parts.append(
                types.Part(
                    function_response=types.FunctionResponse(
                        id=call.id,          # may be None; that is fine
                        name=call.name,
                        response={"result": result},
                    )
                )
            )

        contents.append(types.Content(role="user", parts=result_parts))

    print("Stopped: maximum steps reached.")
    return ""


In [10]:
questions = [
    "What are the ingredients needed to make golibaje?",
    "How long should the mangalore buns dough rest?",
    "Does the neer dosa batter need fermentation?",
    "What is the most famous street food in Mumbai?",   # not in documents
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(20)   # stay inside the free-tier rate limit
    print("\n" + "=" * 60)
    run_agent(q)


Question: What are the ingredients needed to make golibaje?
--------------------------------------------------
Step 1: search_documents({'query': 'golibaje'})
  Result preview: [Source: golibaje]
1 cup maida (all-purpose flour). 1/2 cup curd. 2 tablespoons rice flour. 1 tables...

Final answer:
Based on the recipe, the ingredients needed to make golibaje are:

* **Maida (all-purpose flour):** 1 cup
* **Curd:** 1/2 cup
* **Rice flour:** 2 tablespoons
* **Sugar:** 1 tablespoon
* **Cumin seeds:** 1/2 teaspoon
* **Baking soda:** 1/2 teaspoon
* **Green chilli:** 1, finely chopped
* **Coriander:** 1 tablespoon, chopped
* **Ginger:** 1/2 teaspoon, grated
* **Salt:** To taste
* **Water:** As required
* **Oil:** For deep frying

Question: How long should the mangalore buns dough rest?
--------------------------------------------------
Step 1: search_documents({'query': 'mangalore buns dough rest'})
  Result preview: [Source: mangalore_buns]
2 cups maida (all-purpose flour). 2 ripe bananas. 1/4

In [11]:
# This cell will check if you have added atleast 2 more questions.
assert(len(questions) >= 4)
# This will check if you added a question.
assert("Add a question here." not in questions)